# ML-09 — Validation and Research Claim Audit

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/vishalseries7030/flyrank-ml-internship/blob/main/work/notebooks/w06_validation_audit.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

### Finding 1: Growing vs. declining content

The FlyRank report observed that growing pages were younger on average than declining pages: approximately 185 days versus 228 days. Average word count was almost identical (1,487 versus 1,481 words), suggesting that age showed a clearer difference than length in this comparison.

**Where does the label come from?** The report defines trend direction by comparing the latest 30 days with the previous 30 days. This is an observed trend label, not proof of future growth.

**Methodology question:** Were the groups evaluated on a later, untouched time period, and were repeated pages or clients kept separate during validation? Without that evidence, the comparison supports an association in the observed data, not a claim that age causes decline.

### Finding 2: Freshness and growth-to-decline ratio

The report observed a growth-to-decline ratio of about 5.43:1 for pages updated 31–90 days earlier. However, the reported ratio for pages in the 361+ day freshness bucket was much larger and based on a small group, including only 21 declining pages.

**Where does the label come from?** The outcome is the reported growth or decline category based on trend direction, rather than a randomized experiment assigning pages to refresh dates.

**Methodology question:** Were differences in page age, existing visibility, and editorial selection accounted for? A higher growth-to-decline ratio alone does not show that refreshing a page caused growth. The findings are useful for prioritizing further review, but need validation on an appropriate holdout before stronger predictive claims are made.

In [7]:
# Reproduce the two reported comparisons and flag the small subgroup.
growing_age_days = 185
declining_age_days = 228
growing_word_count = 1487
declining_word_count = 1481

freshness_31_90_ratio = 5.43
freshness_361_plus_declining_pages = 21

print("Finding 1: Growing vs. declining")
print("Age difference (days):", declining_age_days - growing_age_days)
print(
    "Growing pages are younger by (%):",
    round((declining_age_days - growing_age_days) / declining_age_days * 100, 1)
)
print("Word-count difference:", growing_word_count - declining_word_count)

print("\nFinding 2: Freshness")
print("Reported growth-to-decline ratio (31–90 days):", freshness_31_90_ratio)
print("Declining pages in 361+ day bucket:",
      freshness_361_plus_declining_pages)

assert growing_age_days < declining_age_days
assert abs(growing_word_count - declining_word_count) < 20
assert freshness_361_plus_declining_pages < 30

print("\nChecks passed. These are reported descriptive findings, not causal proof.")

Finding 1: Growing vs. declining
Age difference (days): 43
Growing pages are younger by (%): 18.9
Word-count difference: 6

Finding 2: Freshness
Reported growth-to-decline ratio (31–90 days): 5.43
Declining pages in 361+ day bucket: 21

Checks passed. These are reported descriptive findings, not causal proof.


## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

### Week-5 model validation: grouped split

I evaluated the Logistic Regression classifier using a client-grouped 80/20 train/test split. The split held out 7 clients for testing and had zero client overlap between training and testing.

On this grouped test set, Logistic Regression achieved Average Precision of 0.5790, ROC-AUC of 0.5841, and Precision@20 of 0.85. The ML-07 rule-based baseline achieved 0.5124, 0.5048, and 0.35 on the same test rows, respectively.

An important limitation is that all 6,163 test rows are keyword articles, while the training data also contains Feedly and comparison articles. Therefore, these metrics describe performance on keyword articles from held-out clients and should not be generalized to all content types.

The current results do not include the original non-grouped Week-5 metrics, so I will not claim a measured improvement from changing the split. These results measure agreement with the existing trend label, not future decline or the causal impact of refreshing content.

In [8]:
# ML-09: Record and validate the observed grouped-split results.
# These values come from the completed ML-08 run above.

grouped_results = pd.DataFrame([
    {
        "Approach": "ML-07 rule-based baseline",
        "Average Precision": 0.5124,
        "ROC-AUC": 0.5048,
        "Precision@20": 0.35,
    },
    {
        "Approach": "ML-08 Logistic Regression",
        "Average Precision": 0.5790,
        "ROC-AUC": 0.5841,
        "Precision@20": 0.85,
    },
])

display(grouped_results.round(4))

print("Training clients:", 25)
print("Held-out test clients:", 7)
print("Client overlap:", 0)
print("Test rows:", 6163)
print("Test content types: keyword articles only")

assert grouped_results.shape[0] == 2
assert grouped_results["Average Precision"].between(0, 1).all()
assert grouped_results["ROC-AUC"].between(0, 1).all()
assert grouped_results["Precision@20"].between(0, 1).all()

print("\nPASS: grouped-split results recorded and checked.")
print("Limitation: original non-grouped metrics are not available here.")

,Approach,Average Precision,ROC-AUC,Precision@20
0,ML-07 rule-based baseline,0.5124,0.5048,0.35
1,ML-08 Logistic Regression,0.5790,0.5841,0.85


Training clients: 25
Held-out test clients: 7
Client overlap: 0
Test rows: 6163
Test content types: keyword articles only

PASS: grouped-split results recorded and checked.
Limitation: original non-grouped metrics are not available here.


In [9]:
import pandas as pd

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

### Leakage audit

The feature set excludes direct identifiers (`content_id`, `client_id`) and target-related fields (`trend_direction`, `trend_pct`). It also excludes the recent-versus-previous 30-day metrics used to construct the decline label, reducing the risk of direct label leakage.

However, excluding these fields does not prove the model is completely leakage-free. Several included features—such as `sessions_90d`, `clicks_90d`, `impressions_90d`, `ctr`, `avg_position`, `engagement_rate`, `scroll_rate`, `ai_traffic_pct`, `provider_used`, and `model_used`—require a timing and provenance check. I need to confirm that these features were available at the intended prediction time and were not derived from information overlapping with the target-label window.

The client-grouped split prevents the same clients from appearing in both training and test sets. It does not independently prevent temporal leakage or establish that the model predicts future declines. The test set also contains only keyword articles, so these results should not be generalized to other content types.

**Conclusion:** The current feature exclusions reduce obvious leakage risks, but the model should be described as predicting the existing decline proxy until feature timing and a genuinely forward-looking evaluation are verified.

In [10]:
# ML-09: explicit feature and leakage audit

target_fields = {
    "trend_direction",
    "trend_pct",
    "target_declining_proxy",
}

identifier_fields = {
    "content_id",
    "client_id",
}

label_window_fields = {
    "last_30d_clicks",
    "previous_30d_clicks",
    "last_30d_impressions",
    "previous_30d_impressions",
}

included_features = {
    "search_volume", "competition", "cpc",
    "word_count", "char_count",
    "impressions_90d", "clicks_90d",
    "pageviews_90d", "sessions_90d",
    "users_90d", "engaged_sessions_90d",
    "ai_sessions_90d", "scroll_events_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate",
    "scroll_rate", "ai_traffic_pct",
    "content_type", "main_intent", "competition_level",
    "age_tier", "freshness_tier", "word_count_tier",
    "char_count_tier", "provider_used", "model_used",
}

print("Direct target fields included:",
      sorted(included_features & target_fields))
print("Identifiers included:",
      sorted(included_features & identifier_fields))
print("Known label-window fields included:",
      sorted(included_features & label_window_fields))

assert not (included_features & target_fields)
assert not (included_features & identifier_fields)
assert not (included_features & label_window_fields)

print("\nPASS: no explicitly listed target fields, identifiers, or label-window fields.")
print("REVIEW REQUIRED: confirm the timing and provenance of 90-day metrics,")
print("engagement metrics, and provider/model fields before claiming leakage-free validation.")

Direct target fields included: []
Identifiers included: []
Known label-window fields included: []

PASS: no explicitly listed target fields, identifiers, or label-window fields.
REVIEW REQUIRED: confirm the timing and provenance of 90-day metrics,
engagement metrics, and provider/model fields before claiming leakage-free validation.


## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

### Claim rewrite: from association to a testable claim

**Original claim:** Recently updated content performs better, so refreshing content causes an improvement in performance.

**Rewritten claim:** In the report's descriptive analysis, growing pages were younger on average than declining pages (185 versus 228 days). This is an association, not evidence that refreshing a page causes performance to improve. In my client-grouped evaluation, Logistic Regression achieved Average Precision of 0.5790, ROC-AUC of 0.5841, and Precision@20 of 0.85 on 6,163 test rows from seven held-out clients. The test set contained keyword articles only, and the model predicts a proxy derived from existing trend labels rather than verified future declines.

**What evidence would strengthen the claim?**
1. Verify that every feature was available at the prediction cutoff and did not overlap with the label-construction window.
2. Evaluate on a genuinely later time period while keeping clients separated appropriately.
3. Test performance across the content types expected in deployment.
4. To assess whether refreshing causes improvement, compare refreshed pages with a suitable control group, ideally through a randomized experiment, and measure subsequent outcomes.

**Conclusion:** The current results support a limited claim about classification performance on the existing decline proxy. They do not establish future predictive validity or prove that content refreshes cause improvement.

In [11]:
# ML-09: Claim rewrite audit

original_claim = (
    "Recently updated content performs better, so refreshing content "
    "causes an improvement in performance."
)

rewritten_claim = (
    "Growing pages were younger on average than declining pages "
    "(185 versus 228 days), but this descriptive association does not "
    "prove that refreshing content causes improvement. On a client-grouped "
    "test set of 6,163 keyword-article rows from 7 held-out clients, "
    "Logistic Regression achieved Average Precision of 0.5790, "
    "ROC-AUC of 0.5841, and Precision@20 of 0.85. The model predicts "
    "an existing decline proxy, not verified future declines."
)

evidence_needed = [
    "Verify feature availability at the prediction cutoff.",
    "Evaluate on a genuinely later time period.",
    "Validate across content types expected in deployment.",
    "Use a suitable controlled experiment to test causal refresh impact.",
]

print("Original claim:\n", original_claim)
print("\nRewritten claim:\n", rewritten_claim)
print("\nEvidence needed:")
for i, item in enumerate(evidence_needed, start=1):
    print(f"{i}. {item}")

assert "does not prove" in rewritten_claim
assert "existing decline proxy" in rewritten_claim
assert len(evidence_needed) == 4

print("\nPASS: claim rewritten with limitations and evidence requirements.")

Original claim:
 Recently updated content performs better, so refreshing content causes an improvement in performance.

Rewritten claim:
 Growing pages were younger on average than declining pages (185 versus 228 days), but this descriptive association does not prove that refreshing content causes improvement. On a client-grouped test set of 6,163 keyword-article rows from 7 held-out clients, Logistic Regression achieved Average Precision of 0.5790, ROC-AUC of 0.5841, and Precision@20 of 0.85. The model predicts an existing decline proxy, not verified future declines.

Evidence needed:
1. Verify feature availability at the prediction cutoff.
2. Evaluate on a genuinely later time period.
3. Validate across content types expected in deployment.
4. Use a suitable controlled experiment to test causal refresh impact.

PASS: claim rewritten with limitations and evidence requirements.


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.